In [16]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence


In [1]:
import pandas as pd

from read_data import ImportTrainingData

X_events, y = ImportTrainingData()

In [8]:
def collate_sequence_batch(batch):
    """LLM assisted (UiOGPT)"""
    """
    Padder en batch med variable sekvenser.

    Returnerer
    ----------
    x_padded : torch.Tensor
        Shape: (batch_size, maks_sekvenslengde_i_batch, antall_features)

    lengths : torch.Tensor
        Shape: (batch_size,)
        Faktiske lengder før padding.

    y : torch.Tensor
        Shape: (batch_size,)
    """
    sequences, targets = zip(*batch)

    lengths = torch.tensor(
        [len(sequence) for sequence in sequences],
        dtype=torch.long,
    )

    # batch_first=True gir:
    # (batch_size, maks_lengde, antall_features)
    x_padded = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=0.0,
    )

    y = torch.stack(targets)

    return x_padded, lengths, y


In [ ]:
class SequenceDataFrameDataset(Dataset):
    def __init__(
        self,
        dataframes,
        targets=None,
        object_type_column="c_object_type",
        categories=None,
        dtype=torch.float32,
    ):
        """LLM assisted (UiOGPT)"""
        """
        Parametere
        ----------
        dataframes : list[pd.DataFrame]
            Ett DataFrame per sekvens.
            Hver rad er ett tidsteg.

        targets : array-like eller None
            Én target per DataFrame/sekvens.
            For eksempel en liste med risikoer.

        object_type_column : str
            Kolonnen som skal one-hot-encodes.

        categories : list[str] eller None
            Kategorier for c_object_type.
            Dersom None: finn kategorier fra dataframes.
            Ved validering/test: send inn kategoriene fra treningssettet.

        dtype : torch.dtype
            Datatype for input og target.
        """
        if len(dataframes) == 0:
            raise ValueError("dataframes kan ikke være tom.")

        if targets is not None and len(dataframes) != len(targets):
            raise ValueError(
                "dataframes og targets må ha samme antall elementer."
            )

        self.dataframes = dataframes
        self.targets = targets
        self.object_type_column = object_type_column
        self.dtype = dtype

        # Kontroller kolonner og hent numeriske feature-kolonner
        first_df = dataframes[0]

        if object_type_column not in first_df.columns:
            raise KeyError(
                f"Kolonnen '{object_type_column}' finnes ikke."
            )

        self.original_columns = list(first_df.columns)

        self.numeric_columns = [
            col for col in self.original_columns
            if col != object_type_column
        ]

        for i, df in enumerate(dataframes):
            if len(df) == 0:
                raise ValueError(f"DataFrame med indeks {i} er tomt.")

            if list(df.columns) != self.original_columns:
                raise ValueError(
                    f"DataFrame {i} har andre eller feil sorterte kolonner."
                )

        # Finn kategorier fra første rad i hvert DataFrame.
        # Kun nødvendig fordi c_object_type er konstant i en sekvens.
        if categories is None:
            categories = sorted({
                str(df[object_type_column].iloc[0])
                for df in dataframes
            })

        self.categories = list(categories)

        self.category_to_index = {
            category: index
            for index, category in enumerate(self.categories)
        }

        self.feature_names = (
            self.numeric_columns
            + [f"{object_type_column}__{category}"
               for category in self.categories]
        )

    def __len__(self):
        return len(self.dataframes)

    def __getitem__(self, index):
        df = self.dataframes[index]

        # Numeriske kolonner: shape = (sekvenslengde, antall_numeriske_features)
        numeric_values = df[self.numeric_columns].to_numpy(
            dtype=np.float32
        )

        sequence_length = len(df)

        # Leser bare første objekt-type, ettersom den er lik for hele sekvensen
        object_type = str(df[self.object_type_column].iloc[0])

        # Dersom en ukjent kategori dukker opp i validering/test,
        # får den en ren nullvektor.
        one_hot = np.zeros(
            (sequence_length, len(self.categories)),
            dtype=np.float32,
        )

        if object_type in self.category_to_index:
            category_index = self.category_to_index[object_type]
            one_hot[:, category_index] = 1.0
        else:
            print(
                f"Advarsel: Ukjent c_object_type '{object_type}' "
                f"i DataFrame {index}."
            )

        # Kombinerer numeriske og one-hot-features
        sequence = np.concatenate(
            [numeric_values, one_hot],
            axis=1,
        )

        # PyTorch-tensor med shape:
        # (sekvenslengde, antall_features)
        x = torch.tensor(sequence, dtype=self.dtype)

        if self.targets is None:
            return x

        # Én target per sekvens, f.eks. risiko
        y = torch.tensor(
            self.targets[index],
            dtype=self.dtype,
        )

        return x, y


In [ ]:
#Skrevet av ChatGPT:


class StackedRNN(nn.Module):
    """LLM assisted"""

    def __init__(self):
        """LLM assisted"""
        super().__init__()

        # RNN-lag 1: 3 input-features -> 16 hidden units
        self.rnn1 = nn.RNN(
            input_size=105,
            hidden_size=30,
            batch_first=True
        )

        # RNN-lag 2: 16 input-features -> 10 hidden units
        self.rnn2 = nn.RNN(
            input_size=30,
            hidden_size=10,
            batch_first=True
        )

        # RNN-lag 3: 10 input-features -> 5 hidden units
        self.rnn3 = nn.RNN(
            input_size=10,
            hidden_size=5,
            batch_first=True
        )

        # Output-lag: 5 -> 2 klasser
        self.fc = nn.Linear(5, 1)

    def forward(self, x, lengths):
        """LLM assisted"""

        packed_x = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first = True,
            enforce_sorted = False
        )

        # Første RNN-lag
        packed_out1, hidden1 = self.rnn1(packed_x)

        # Andre RNN-lag får output fra det første
        packed_out2, hidden2 = self.rnn2(packed_out1)

        # Tredje RNN-lag får output fra det andre
        packed_out3, hidden3 = self.rnn3(packed_out2)

        # Hent siste hidden state fra tredje RNN-lag
        final_hidden = hidden3[-1]

        # Lag prediksjon
        prediction = self.fc(final_hidden)

        return prediction.squeeze(-1)


In [9]:
class F2VektetMSELoss(nn.Module):
    """LLM assisted (UiOGPT)"""
    def __init__(self, threshold=1e-6, beta=2.0, eps=1e-8):
        """LLM assisted (UiOGPT)"""
        super().__init__()
        self.threshold = threshold
        self.beta = beta
        self.eps = eps

    def beregn_f2(self, y_true, y_pred):
        """LLM assisted (UiOGPT)"""
        """
        Beregner F2-score basert på hele inputen.

        Positiv klasse:
            risiko >= threshold
        """
        y_true = y_true.reshape(-1)
        y_pred = y_pred.reshape(-1)

        ekte_positiv = y_true >= self.threshold
        predikert_positiv = y_pred >= self.threshold

        tp = (ekte_positiv & predikert_positiv).sum().float()
        fp = (~ekte_positiv & predikert_positiv).sum().float()
        fn = (ekte_positiv & ~predikert_positiv).sum().float()

        precision = tp / (tp + fp + self.eps)
        recall = tp / (tp + fn + self.eps)

        beta_kvadrat = self.beta ** 2

        f2 = (
            (1 + beta_kvadrat) * precision * recall
            / (beta_kvadrat * precision + recall + self.eps)
        )

        return f2

    def forward(self, y_pred, y_true, f2_score=None):
        """LLM assisted (UiOGPT)"""
        """
        Parameters
        ----------
        y_pred : torch.Tensor
            Predikert risiko, r_hat.

        y_true : torch.Tensor
            Fasiten / faktisk risiko, r.

        f2_score : float eller torch.Tensor, optional
            F2 beregnet over hele datasettet. Hvis ikke gitt,
            beregnes F2 fra batchen. For trening bør du helst
            sende inn en F2-score fra hele treningssettet.
        """
        y_pred = y_pred.reshape(-1)
        y_true = y_true.reshape(-1)

        # MSE skal bare gjelde den høye risikoklassen
        high_risk_mask = y_true >= self.threshold

        # Dersom batchen ikke har noen positive eksempler
        if not high_risk_mask.any():
            return y_pred.sum() * 0.0

        mse = torch.mean(
            (y_true[high_risk_mask] - y_pred[high_risk_mask]) ** 2
        )

        if f2_score is None:
            # detach fordi F2 bruker hard terskel og ikke er deriverbar
            with torch.no_grad():
                f2_score = self.beregn_f2(y_true, y_pred)

        if not torch.is_tensor(f2_score):
            f2_score = torch.tensor(
                f2_score,
                dtype=y_pred.dtype,
                device=y_pred.device,
            )
        else:
            f2_score = f2_score.to(
                dtype=y_pred.dtype,
                device=y_pred.device,
            )

        # Unngår divisjon på null
        f2_score = torch.clamp(f2_score, min=self.eps)

        loss = mse / f2_score

        return loss


In [26]:
import numpy as np
import pandas as pd

X_events_clean = []

for df in X_events:
    clean_df = df.copy()

    # Gjør +inf og -inf til NaN.
    clean_df = clean_df.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    # Finn bare numeriske kolonner.
    numeric_columns = clean_df.select_dtypes(
        include=[np.number]
    ).columns

    # En enkel teknisk imputering:
    # fyll manglende verdier innenfor samme sekvens.
    clean_df[numeric_columns] = (
        clean_df[numeric_columns]
        .ffill()
        .bfill()
        .fillna(0.0)
    )

    X_events_clean.append(clean_df)


In [33]:
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

start_time = time.time()

print("Starter preprocessing ...", flush=True)

model = StackedRNN()

det_columns = [
    "t_position_covariance_det",
    "c_position_covariance_det",
]

X_events_model = []
n_sequences = len(X_events)

print(f"Antall sekvenser: {n_sequences}", flush=True)

# ---------------------------------------------------
# Steg 1: Rens Inf og log-transformer determinantene
# ---------------------------------------------------
for i, df in enumerate(X_events):
    model_df = df.copy()

    model_df = model_df.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    for column in det_columns:
        values = model_df[column].to_numpy(dtype=np.float64)

        values = np.where(
            np.isnan(values),
            np.nan,
            np.clip(values, a_min=0.0, a_max=None),
        )

        model_df[column] = np.log1p(values)

    X_events_model.append(model_df)

    # Skriv ut omtrent 20 oppdateringer totalt.
    if (i + 1) % max(1, n_sequences // 20) == 0 or i == n_sequences - 1:
        print(
            f"Transformert {i + 1}/{n_sequences} sekvenser "
            f"({100 * (i + 1) / n_sequences:.1f} %)",
            flush=True,
        )

print(
    f"Transformasjon ferdig etter {time.time() - start_time:.1f} sekunder.",
    flush=True,
)

# ---------------------------------------------------
# Steg 2: Finn numeriske kolonner
# ---------------------------------------------------
print("Finner numeriske kolonner ...", flush=True)

numeric_columns = X_events_model[0].select_dtypes(
    include=[np.number]
).columns.tolist()

print(
    f"Fant {len(numeric_columns)} numeriske kolonner.",
    flush=True,
)

# ---------------------------------------------------
# Steg 3: Samle data og beregne medianer
# ---------------------------------------------------
# Dette kan være det tregeste/minnekrevende steget.
print("Samler numeriske data for medianberegning ...", flush=True)

all_numeric = pd.concat(
    [df[numeric_columns] for df in X_events_model],
    ignore_index=True,
)

print(
    f"Samlet DataFrame har shape: {all_numeric.shape}",
    flush=True,
)

print("Beregner medianer for imputering ...", flush=True)

fill_values = all_numeric.median()

print(
    f"Medianer beregnet etter {time.time() - start_time:.1f} sekunder.",
    flush=True,
)

# Du trenger ikke beholde den store samlede DataFrame-en videre.
del all_numeric

# ---------------------------------------------------
# Steg 4: Fyll NaN
# ---------------------------------------------------
print("Fyller NaN-verdier ...", flush=True)

for i, df in enumerate(X_events_model):
    df[numeric_columns] = df[numeric_columns].fillna(fill_values)

    if (i + 1) % max(1, n_sequences // 20) == 0 or i == n_sequences - 1:
        print(
            f"Fylt {i + 1}/{n_sequences} sekvenser "
            f"({100 * (i + 1) / n_sequences:.1f} %)",
            flush=True,
        )

print(
    f"NaN-imputering ferdig etter {time.time() - start_time:.1f} sekunder.",
    flush=True,
)

# ---------------------------------------------------
# Steg 5: Dataset og DataLoader
# ---------------------------------------------------
print("Oppretter Dataset og DataLoader ...", flush=True)

dataset = SequenceDataFrameDataset(
    dataframes=X_events_model,
    targets=y,
)

train_loader = DataLoader(
    dataset,
    shuffle=True,
    batch_size=100,
    collate_fn=collate_sequence_batch,
)

print(
    f"Dataset klart: {len(dataset)} sekvenser, "
    f"{len(train_loader)} batcher per epoch.",
    flush=True,
)

# Kontroller én batch.
print("Henter og kontrollerer én batch ...", flush=True)

batch_X, lengths, batch_y = next(iter(train_loader))

print("batch_X shape:", batch_X.shape, flush=True)
print("NaN i X:", torch.isnan(batch_X).any().item(), flush=True)
print("Inf i X:", torch.isinf(batch_X).any().item(), flush=True)
print("NaN i y:", torch.isnan(batch_y).any().item(), flush=True)
print("Inf i y:", torch.isinf(batch_y).any().item(), flush=True)

# Ikke start trening før dette er oppfylt.
assert not torch.isnan(batch_X).any(), "batch_X inneholder fortsatt NaN."
assert not torch.isinf(batch_X).any(), "batch_X inneholder fortsatt Inf."



Starter preprocessing ...
Antall sekvenser: 8293
Transformert 414/8293 sekvenser (5.0 %)
Transformert 828/8293 sekvenser (10.0 %)
Transformert 1242/8293 sekvenser (15.0 %)
Transformert 1656/8293 sekvenser (20.0 %)
Transformert 2070/8293 sekvenser (25.0 %)
Transformert 2484/8293 sekvenser (30.0 %)
Transformert 2898/8293 sekvenser (34.9 %)
Transformert 3312/8293 sekvenser (39.9 %)
Transformert 3726/8293 sekvenser (44.9 %)
Transformert 4140/8293 sekvenser (49.9 %)
Transformert 4554/8293 sekvenser (54.9 %)
Transformert 4968/8293 sekvenser (59.9 %)
Transformert 5382/8293 sekvenser (64.9 %)
Transformert 5796/8293 sekvenser (69.9 %)
Transformert 6210/8293 sekvenser (74.9 %)
Transformert 6624/8293 sekvenser (79.9 %)
Transformert 7038/8293 sekvenser (84.9 %)
Transformert 7452/8293 sekvenser (89.9 %)
Transformert 7866/8293 sekvenser (94.9 %)
Transformert 8280/8293 sekvenser (99.8 %)
Transformert 8293/8293 sekvenser (100.0 %)
Transformasjon ferdig etter 10.8 sekunder.
Finner numeriske kolonner ..

In [34]:
# Opprett modellen ETTER datasettet, med korrekt input-størrelse.
model = StackedRNN()

threshold = 1e-5
beta = 2.0
eps = 0.1

criterion = F2VektetMSELoss(
    beta=beta,
    threshold=threshold,
    eps=eps,
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
)

# ---------------------------------------------------
# Steg 6: Trening
# ---------------------------------------------------
epochs = 20

print("Starter trening ...", flush=True)

for epoch in range(epochs):
    epoch_start = time.time()

    model.train()
    total_loss = 0.0

    for batch_index, (batch_X, lengths, batch_y) in enumerate(train_loader):
        optimizer.zero_grad()

        predictions = model(batch_X, lengths)
        loss = criterion(predictions, batch_y)

        # Stopp med forklarende feil i stedet for å fortsette med NaN.
        if not torch.isfinite(loss):
            raise ValueError(
                f"Loss er ikke endelig i epoch {epoch + 1}, "
                f"batch {batch_index + 1}. Loss: {loss.item()}"
            )

        loss.backward()

        # Anbefalt for RNN.
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        total_loss += loss.item()

        # Vis progresjon omtrent ti ganger per epoch.
        if (batch_index + 1) % max(1, len(train_loader) // 10) == 0:
            print(
                f"Epoch {epoch + 1}/{epochs} | "
                f"batch {batch_index + 1}/{len(train_loader)} | "
                f"loss: {loss.item():.6f}",
                flush=True,
            )

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{epochs} ferdig | "
        f"gjennomsnittlig loss: {average_loss:.6f} | "
        f"tid: {time.time() - epoch_start:.1f} sekunder",
        flush=True,
    )


Starter trening ...
Epoch 1/20 | batch 8/83 | loss: 4.345909
Epoch 1/20 | batch 16/83 | loss: 0.000000
Epoch 1/20 | batch 24/83 | loss: 3.235540
Epoch 1/20 | batch 32/83 | loss: 3.592696
Epoch 1/20 | batch 40/83 | loss: 2.460371
Epoch 1/20 | batch 48/83 | loss: 5.108070
Epoch 1/20 | batch 56/83 | loss: 0.000000
Epoch 1/20 | batch 64/83 | loss: 0.000000
Epoch 1/20 | batch 72/83 | loss: 2.142784
Epoch 1/20 | batch 80/83 | loss: 1.638538
Epoch 1/20 ferdig | gjennomsnittlig loss: 2.016791 | tid: 12.4 sekunder
Epoch 2/20 | batch 8/83 | loss: 0.000000
Epoch 2/20 | batch 16/83 | loss: 2.936034
Epoch 2/20 | batch 24/83 | loss: 0.000000
Epoch 2/20 | batch 32/83 | loss: 0.412350
Epoch 2/20 | batch 40/83 | loss: 0.276630
Epoch 2/20 | batch 48/83 | loss: 0.125418
Epoch 2/20 | batch 56/83 | loss: 0.022566
Epoch 2/20 | batch 64/83 | loss: 0.001613
Epoch 2/20 | batch 72/83 | loss: 0.000000
Epoch 2/20 | batch 80/83 | loss: 0.000000
Epoch 2/20 ferdig | gjennomsnittlig loss: 0.230023 | tid: 11.5 sekunde